### Ingest Race Data
Fetches race data from the Jolpica F1 API for a given season and round,
and writes structured files to the landing volume for downstream bronze ingestion.

**Outputs per batch_id folder:**
| File | Format | Notes |
|---|---|---|
| `results/results_{season}.json` | NDJSON | One record per driver result |
| `sprints/sprints_{season}.json` | JSON array | Empty array `[]` for non-sprint rounds |
| `circuits.csv` | CSV | Full circuit reference (all-time) |
| `races.csv` | CSV | Full race schedule reference (all-time) |
| `constructors.json` | NDJSON | Full constructor reference (all-time) |
| `drivers.json` | NDJSON | Full driver reference (all-time) |

In [0]:
dbutils.widgets.text("p_season", "")
dbutils.widgets.text("p_round_no", "")

In [0]:
v_season   = dbutils.widgets.get("p_season")
v_round_no = dbutils.widgets.get("p_round_no")

In [0]:
%run ../00-common/01.environment-config

In [0]:
from ingestion_helpers import validate_batch_inputs, fetch_paginated, parse_results, parse_sprints, parse_drivers
from datetime import date, datetime, timedelta

In [0]:
CHECK_WINDOW_DAYS = 6

In [0]:
# Partial override → fail loudly rather than silently falling back to
# auto-detect, which would otherwise mask a typo or incomplete parameter set.
if bool(v_season) != bool(v_round_no):
    raise ValueError(
        "Partial override detected: both p_season and p_round_no must be "
        "provided together for a manual backfill, or both left blank for an automated run. "
        f"Got p_season='{v_season}', p_round_no='{v_round_no}'."
    )

In [0]:
if v_season and v_round_no:
    # Manual override — backfill path
    print(f"Manual override: season={v_season}, round={v_round_no}")

else:
    # Auto-detect path — normal automated run
    today           = date.today()
    season_to_check = str(today.year)

    schedule = fetch_paginated(f"{season_to_check}/races/", "RaceTable", "Races")

    match = None
    for race in schedule:
        race_date = datetime.strptime(race["date"], "%Y-%m-%d").date()
        if today - timedelta(days=CHECK_WINDOW_DAYS) <= race_date < today:
            match = race
            break

    if match is None:
        dbutils.notebook.exit(
            f"No race in the last {CHECK_WINDOW_DAYS} days — not a race week, exiting."
        )

    v_season, v_round_no = season_to_check, match["round"]
    print(f"Auto-detected: season={v_season}, round={v_round_no} "
          f"({match['raceName']} on {match['date']})")

In [0]:
batch_id = validate_batch_inputs(v_season, v_round_no)
print(f"batch_id : {batch_id}")

In [0]:
import requests
import json
import os
import csv

In [0]:
BASE_URL  = "https://api.jolpi.ca/ergast/f1"
batch_dir = os.path.join(landing_folder_path, batch_id)

### Results

In [0]:
results_response = requests.get(f"{BASE_URL}/{v_season}/{v_round_no}/results/")
results_response.raise_for_status()

results_records = parse_results(results_response.json()["MRData"]["RaceTable"]["Races"])

results_dir = os.path.join(batch_dir, "results")
os.makedirs(results_dir, exist_ok=True)

results_file = os.path.join(results_dir, f"results_{v_season}.json")
with open(results_file, "w") as f:
    for record in results_records:
        f.write(json.dumps(record) + "\n")

print(f"Results  : {len(results_records)} records, {results_file}")

### Sprints

In [0]:
sprint_response = requests.get(f"{BASE_URL}/{v_season}/{v_round_no}/sprint/")
sprint_response.raise_for_status()

sprint_records = parse_sprints(sprint_response.json()["MRData"]["RaceTable"]["Races"])

if not sprint_records:
    print("Sprints  : no sprint this round — writing empty file")

sprints_dir = os.path.join(batch_dir, "sprints")
os.makedirs(sprints_dir, exist_ok=True)

sprints_file = os.path.join(sprints_dir, f"sprints_{v_season}.json")
with open(sprints_file, "w") as f:
    json.dump(sprint_records, f, indent=2)

print(f"Sprints  : {len(sprint_records)} records, {sprints_file}")

### Circuits

In [0]:
circuits_response = requests.get(f"{BASE_URL}/circuits/?limit=100")
circuits_response.raise_for_status()

circuits_records = []
for circuit in circuits_response.json()["MRData"]["CircuitTable"]["Circuits"]:
    circuits_records.append({
        "circuitId":   circuit["circuitId"],
        "url":         circuit["url"],
        "circuitName": circuit["circuitName"].lower(),
        "lat":         circuit["Location"]["lat"],
        "long":        circuit["Location"]["long"],
        "locality":    circuit["Location"]["locality"].lower(),
        "country":     circuit["Location"]["country"]
    })

circuits_file = os.path.join(batch_dir, "circuits.csv")
with open(circuits_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["circuitId", "url", "circuitName", "lat", "long", "locality", "country"])
    writer.writeheader()
    writer.writerows(circuits_records)

print(f"Circuits : {len(circuits_records)} records, {circuits_file}")

### Races

In [0]:
races_raw = fetch_paginated("races/", "RaceTable", "Races")

races_records = []
for race in races_raw:
    races_records.append({
        "season":    race["season"],
        "round":     race["round"],
        "url":       race["url"],
        "raceName":  race["raceName"].lower(),
        "date":      race["date"],
        "circuitId": race["Circuit"]["circuitId"]
    })

races_file = os.path.join(batch_dir, "races.csv")
with open(races_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["season", "round", "url", "raceName", "date", "circuitId"])
    writer.writeheader()
    writer.writerows(races_records)

print(f"Races    : {len(races_records)} records, {races_file}")

### Constructors

In [0]:
constructors_raw = fetch_paginated("constructors/", "ConstructorTable", "Constructors")

constructors_records = []
for constructor in constructors_raw:
    constructors_records.append({
        "constructorId": constructor["constructorId"],
        "name":          constructor["name"],
        "nationality":   constructor["nationality"].lower(),
        "url":           constructor.get("url", "")
    })

constructors_file = os.path.join(batch_dir, "constructors.json")
with open(constructors_file, "w") as f:
    for record in constructors_records:
        f.write(json.dumps(record) + "\n")

print(f"Constructors: {len(constructors_records)} records, {constructors_file}")

### Drivers

In [0]:
drivers_raw     = fetch_paginated("drivers/", "DriverTable", "Drivers")
drivers_records = parse_drivers(drivers_raw)

drivers_file = os.path.join(batch_dir, "drivers.json")
with open(drivers_file, "w") as f:
    for record in drivers_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Drivers  : {len(drivers_records)} records, {drivers_file}")

In [0]:
print(f"\nbatch_id '{batch_id}' successfully written to landing.")